# ⚡ asyncio + Python Concurrency — Days 11–14 Code-Along

**Sprint:** 14-Day AI Engineer Skills Sprint  
**Topic:** asyncio fundamentals → async LLM calls → async LangGraph agent  
**Pace:** ~2 hrs/day | Style: short concept read → heavy hands-on  

---

## How to use this notebook

- Each day has a **Concepts** section — read it even on the laziest day. It has the key mental models.  
- Each day has **📖 Must-Read Docs** — skim these before coding.  
- Each day has **🧪 Try It Yourself** cells — attempt these yourself first.  
- Each day has **✅ Reference Solution** cells — complete, working code to fall back on.  
- Day 12 and Day 14 have LinkedIn post reminders.  

---

## Important: async in Jupyter

Jupyter notebooks already run an event loop — so you do **NOT** call `asyncio.run()` inside cells.  
Just use `await` directly at the top level of a cell.  
When you need `asyncio.run()` (e.g. in a `.py` script), the pattern is shown separately.

---

## Environment

- Python 3.10+  
- OpenAI API key (or your Gemini/Bedrock adapter)  
- For Day 12 FastAPI: run the server from a terminal, not this notebook  

In [ ]:
# ── SETUP: Install dependencies ──────────────────────────────────────────────
# Run this once before starting Day 11.

# !pip install -U langchain-openai langgraph fastapi uvicorn httpx tenacity python-dotenv

import sys, asyncio, os
from dotenv import load_dotenv

load_dotenv()

import langchain_openai, langgraph
print(f"Python          : {sys.version.split()[0]}")
print(f"asyncio version : {asyncio.__version__}")
print(f"langchain-openai: {langchain_openai.__version__}")
print(f"langgraph       : {langgraph.__version__}")
print(f"OPENAI_API_KEY  : {'set' if os.getenv('OPENAI_API_KEY') else 'NOT SET'}")

---
# 📅 Day 11 — How asyncio Actually Works

**Time budget:** ~40 min concepts + 80 min hands-on

---

## 📖 Must-Read Docs

| Doc | What you'll learn | URL |
|---|---|---|
| Python asyncio docs | Event loop, coroutines, tasks | https://docs.python.org/3/library/asyncio.html |
| Real Python asyncio guide | Deep conceptual walkthrough | https://realpython.com/async-io-python/ |
| asyncio.gather | Fan-out pattern | https://docs.python.org/3/library/asyncio-task.html#asyncio.gather |

**Minimum read on a lazy day:** Read the Real Python asyncio intro through the "async/await" section. The event loop diagram will rewire how you think about concurrency.

---

## 🧠 Core Concepts

### The event loop — what it actually is

```
The event loop is a single-threaded loop that:
  1. Takes a coroutine off its queue
  2. Runs it until it hits an `await`
  3. Parks that coroutine (suspends it)
  4. Picks up the next ready coroutine
  5. Repeats

When the awaited thing (I/O, sleep, network) finishes → the coroutine becomes ready again.
```

This is **cooperative multitasking** — no OS threads, no preemption. The coroutine must `await` to yield control.

### async def / await — what Python does at runtime

```python
async def my_coro():
    result = await some_io()   # ← suspends HERE, event loop runs other tasks
    return result              # ← resumes here when some_io() finishes

# Calling my_coro() does NOT run it — it returns a coroutine object
coro = my_coro()   # just an object, nothing executed yet
result = await coro  # NOW it runs
```

### When asyncio helps vs when it doesn't

```
✅ I/O bound     → network requests, DB queries, file reads, LLM API calls
                   asyncio shines — concurrently wait for many responses

❌ CPU bound     → heavy computation, image processing, ML inference
                   asyncio doesn't help — use ProcessPoolExecutor instead
```

LLM API calls are pure I/O bound → asyncio is ideal.

### The 3 core patterns

```python
# 1. gather() — fan out, wait for ALL to finish (like Promise.all)
results = await asyncio.gather(coro_a(), coro_b(), coro_c())

# 2. create_task() — schedule now, await later (or fire-and-forget)
task = asyncio.create_task(coro_a())
# ... do other work ...
result = await task

# 3. asyncio.timeout() — cancel a coroutine if it takes too long
async with asyncio.timeout(5.0):
    result = await slow_operation()
```

### Common mistakes

```python
# ❌ WRONG: blocking the event loop
async def bad_node(state):
    time.sleep(2)          # blocks the ENTIRE event loop — nothing else runs
    ...                    # use asyncio.sleep(2) instead

# ❌ WRONG: forgetting await
result = some_async_fn()   # returns a coroutine object, not the result!

# ❌ WRONG: mixing sync and async naively
async def bad():
    result = requests.get(url)   # requests is sync — blocks event loop
                                 # use httpx.AsyncClient instead
```

### 🧪 Try It Yourself — 11.1: asyncio.gather() — fan out and collect

Write an `async def fetch_data(name, delay)` that:
- Prints `"Starting: {name}"`
- Awaits `asyncio.sleep(delay)` (simulating I/O)
- Returns `f"{name}: done"`

Then use `asyncio.gather()` to run 5 of them with different delays.  
Print a before/after timestamp to prove they ran concurrently.  
Compare: what would the total time be if run sequentially?

In [ ]:
# 🧪 YOUR CODE HERE — Day 11.1: asyncio.gather() fan-out
# import asyncio, time
# async def fetch_data(name, delay): ...
# In Jupyter: await directly (no asyncio.run needed)

In [ ]:
# ✅ REFERENCE SOLUTION — Day 11.1: gather() fan-out with timing
# ─────────────────────────────────────────────────────────────────────────────
# KEY INSIGHTS:
# 1. gather() starts ALL coroutines immediately, then waits for ALL to finish.
#    Total time ≈ max(delays), NOT sum(delays). This is the whole point.
# 2. Results are returned in the SAME ORDER as the coroutines passed in,
#    regardless of which finishes first. Safe to unpack positionally.
# 3. In Jupyter: don't use asyncio.run() — just `await` at the cell level.
#    In a .py script: wrap in `async def main(): ... asyncio.run(main())`

import asyncio
import time

async def fetch_data(name: str, delay: float) -> str:
    """Simulate an I/O-bound operation (network call, DB query, LLM API call)."""
    print(f"  → Starting : {name} (will take {delay}s)")
    await asyncio.sleep(delay)   # ← suspends here, event loop runs others
    print(f"  ✓ Finished : {name}")
    return f"{name}: done after {delay}s"


# Sequential timing — for comparison
tasks = [("worker_A", 1.0), ("worker_B", 0.5), ("worker_C", 1.5), ("worker_D", 0.3), ("worker_E", 0.8)]
sequential_total = sum(d for _, d in tasks)
print(f"Sequential total would be: {sequential_total}s")
print(f"Concurrent total would be: {max(d for _, d in tasks)}s (max delay)")

print("\n=== Running concurrently with gather() ===")
t0 = time.perf_counter()
results = await asyncio.gather(*[fetch_data(name, delay) for name, delay in tasks])
elapsed = time.perf_counter() - t0

print(f"\nElapsed: {elapsed:.2f}s  (sequential would have been {sequential_total:.1f}s)")
print("Results:", results)

### 🧪 Try It Yourself — 11.2: create_task() and cancellation

1. Write `async def slow_task(name, delay)` that sleeps and returns a result
2. Create 3 tasks with `asyncio.create_task()`, do some "other work" (print statements), then await them
3. Create a task and **cancel** it before it finishes — handle the `asyncio.CancelledError`
4. Use `asyncio.timeout(1.0)` to auto-cancel a task that would take 3 seconds

In [ ]:
# 🧪 YOUR CODE HERE — Day 11.2: create_task(), cancellation, timeout

In [ ]:
# ✅ REFERENCE SOLUTION — Day 11.2: Tasks, cancellation, timeout
# ─────────────────────────────────────────────────────────────────────────────
# KEY INSIGHTS:
# 1. create_task() schedules a coroutine NOW but doesn't wait for it.
#    You get a Task object — awaitable, cancellable, queryable (.done(), .result()).
# 2. task.cancel() sends a CancelledError into the coroutine at the next await.
#    The coroutine must not swallow it — re-raise or let it propagate.
# 3. asyncio.timeout() is the clean way to add timeouts in Python 3.11+.
#    It raises asyncio.TimeoutError on expiry — catch it to handle gracefully.

import asyncio

async def slow_task(name: str, delay: float) -> str:
    print(f"  [{name}] started")
    await asyncio.sleep(delay)
    print(f"  [{name}] finished")
    return f"{name} result"


# Pattern 1: create_task — schedule now, await later
print("=== Pattern 1: create_task — fire and continue ===")
task_a = asyncio.create_task(slow_task("A", 0.5))
task_b = asyncio.create_task(slow_task("B", 0.3))
task_c = asyncio.create_task(slow_task("C", 0.8))

print("  (tasks are running in background while we do other work)")
await asyncio.sleep(0.1)   # give them a chance to start
print(f"  task_a done? {task_a.done()}")

results = await asyncio.gather(task_a, task_b, task_c)
print("  Results:", results)


# Pattern 2: cancellation
print("\n=== Pattern 2: cancellation ===")
long_task = asyncio.create_task(slow_task("long_running", 10.0))
await asyncio.sleep(0.1)
long_task.cancel()
try:
    await long_task
except asyncio.CancelledError:
    print("  long_task was cancelled — handled gracefully")


# Pattern 3: asyncio.timeout
print("\n=== Pattern 3: asyncio.timeout ===")
try:
    async with asyncio.timeout(0.5):
        result = await slow_task("timeout_test", 3.0)
        print(f"  result: {result}")  # never reached
except asyncio.TimeoutError:
    print("  Timed out after 0.5s — handled gracefully")

### 🧪 Try It Yourself — 11.3: asyncio.Queue — producer/consumer

Build a producer/consumer pattern:
1. `producer(queue, n)` — puts n items into the queue with a small delay between each
2. `consumer(queue, name)` — reads from queue, processes each item, marks done
3. Run 1 producer + 3 consumers concurrently with `gather()`
4. Use the sentinel value `None` to signal consumers to stop

In [ ]:
# 🧪 YOUR CODE HERE — Day 11.3: asyncio.Queue producer/consumer

In [ ]:
# ✅ REFERENCE SOLUTION — Day 11.3: Producer/consumer with asyncio.Queue
# ─────────────────────────────────────────────────────────────────────────────
# KEY INSIGHTS:
# 1. asyncio.Queue is backpressure-friendly — if maxsize is set, the producer
#    awaits when full (won't overwhelm consumers).
# 2. queue.join() blocks until all items have been get()+task_done().
#    This is the clean shutdown signal — no guessing when consumers are done.
# 3. The sentinel (None) pattern is cleaner for fan-out N consumers:
#    put N Nones, or use queue.join() and then cancel tasks.

import asyncio

NUM_CONSUMERS = 3

async def producer(queue: asyncio.Queue, n: int) -> None:
    """Put n work items into the queue, then send a sentinel per consumer."""
    for i in range(n):
        await asyncio.sleep(0.1)   # simulate work item arriving
        await queue.put(f"item_{i:02d}")
        print(f"  [producer] put item_{i:02d}  (queue size: {queue.qsize()})")
    
    # Send one sentinel per consumer so all consumers stop
    for _ in range(NUM_CONSUMERS):
        await queue.put(None)
    print("  [producer] done")


async def consumer(queue: asyncio.Queue, name: str) -> int:
    """Consume until sentinel received. Return number of items processed."""
    count = 0
    while True:
        item = await queue.get()
        if item is None:
            queue.task_done()
            print(f"  [{name}] received sentinel — exiting")
            break
        await asyncio.sleep(0.2)   # simulate processing time
        print(f"  [{name}] processed {item}")
        count += 1
        queue.task_done()
    return count


print("=== Producer / Consumer pattern ===")
q = asyncio.Queue(maxsize=5)   # backpressure: producer blocks if queue full

prod_task = asyncio.create_task(producer(q, n=9))
cons_tasks = [asyncio.create_task(consumer(q, f"consumer_{i}")) for i in range(NUM_CONSUMERS)]

counts = await asyncio.gather(prod_task, *cons_tasks)
print(f"\nItems processed per consumer: {counts[1:]}")
print(f"Total items processed: {sum(counts[1:])}")

### 🧪 Try It Yourself — 11.4: Calling sync functions safely from async

Sometimes you have a blocking CPU-bound function you can't rewrite.
Use `loop.run_in_executor()` to run it in a thread pool without blocking the event loop:

1. Write a `slow_sync_function(n)` using `time.sleep()` (not asyncio.sleep)
2. Run it synchronously — observe it blocks everything
3. Run it with `asyncio.get_event_loop().run_in_executor(None, fn, *args)` — it no longer blocks
4. Run 5 of them concurrently with `gather()`

In [ ]:
# 🧪 YOUR CODE HERE — Day 11.4: run_in_executor for sync code

In [ ]:
# ✅ REFERENCE SOLUTION — Day 11.4: run_in_executor — bridge sync to async
# ─────────────────────────────────────────────────────────────────────────────
# KEY INSIGHTS:
# 1. run_in_executor(None, fn, *args) runs `fn` in a THREAD POOL.
#    The event loop can process other tasks while the sync code runs in the thread.
#    'None' = use the default ThreadPoolExecutor.
# 2. Use asyncio.to_thread() (Python 3.9+) as a cleaner shorthand.
# 3. For CPU-bound work (not I/O), use ProcessPoolExecutor instead of None
#    to get true parallelism (bypasses the GIL).

import asyncio
import time
from concurrent.futures import ThreadPoolExecutor

def slow_sync_function(n: int) -> str:
    """A blocking function you can't make async (e.g. legacy library)."""
    time.sleep(0.5)   # blocks the OS thread
    return f"sync_result_{n}"


async def run_sync_safely(n: int) -> str:
    """Wrap a sync function so it doesn't block the event loop."""
    loop = asyncio.get_event_loop()
    # run_in_executor offloads to a thread pool
    result = await loop.run_in_executor(None, slow_sync_function, n)
    return result


async def run_sync_with_to_thread(n: int) -> str:
    """Python 3.9+ idiom — cleaner than run_in_executor."""
    return await asyncio.to_thread(slow_sync_function, n)


print("=== 5 sync calls via run_in_executor (concurrent) ===")
t0 = time.perf_counter()
results = await asyncio.gather(*[run_sync_safely(i) for i in range(5)])
elapsed = time.perf_counter() - t0
print(f"Results: {results}")
print(f"Elapsed: {elapsed:.2f}s  (5 × 0.5s sync = {5*0.5}s sequential, {elapsed:.2f}s concurrent)")

print("\n=== Same with asyncio.to_thread ===")
t0 = time.perf_counter()
results2 = await asyncio.gather(*[run_sync_with_to_thread(i) for i in range(5)])
elapsed2 = time.perf_counter() - t0
print(f"Results: {results2}")
print(f"Elapsed: {elapsed2:.2f}s")

### ✅ Day 11 Checkpoint

You should now be able to explain, without looking anything up:
- [ ] Why `gather()` total time ≈ max(delays), not sum(delays)  
- [ ] What `create_task()` does differently from `await` directly  
- [ ] Why `time.sleep()` inside `async def` is dangerous  
- [ ] When to use `run_in_executor` vs `asyncio.to_thread`  
- [ ] What a producer/consumer pattern solves (backpressure, parallel processing)  

**If you can draw the event loop flow for `gather(A, B, C)` on a whiteboard — Day 11 is done.**

---

# 📅 Day 12 — Async LLM Calls: Parallel Requests and Streaming

**Time budget:** ~20 min concepts + 100 min hands-on  
**📝 Write LinkedIn Post #5 tonight** — see `02_linkedin_posts.md`

---

## 📖 Must-Read Docs

| Doc | What you'll learn | URL |
|---|---|---|
| LangChain async | ainvoke, astream, abatch signatures | https://python.langchain.com/docs/concepts/runnables/#async-methods |
| FastAPI StreamingResponse | Server-sent events pattern | https://fastapi.tiangolo.com/advanced/custom-response/#streamingresponse |
| tenacity async | Async retry with exponential backoff | https://tenacity.readthedocs.io/en/latest/#asyncio |

**Minimum read on a lazy day:** Read the LangChain async docs. Understand `ainvoke` vs `astream` and when to use each.

---

## 🧠 Core Concepts

### Why async matters for LLM calls

```
Sequential:  [── call 1 (3s) ──][── call 2 (2s) ──][── call 3 (4s) ──]  = 9s total
Async:       [── call 1 (3s) ──]
             [── call 2 (2s) ──]  = 4s total (max latency wins)
             [── call 3 (4s) ──]
```

For 10 queries: sequential ≈ 47s, async ≈ 8s. Real benchmark from the sprint plan.

### LangChain's async interface

```python
# Async equivalents of sync methods (every Runnable has both)
result = await llm.ainvoke(messages)          # async invoke
async for chunk in llm.astream(messages):     # streaming, token by token
    print(chunk.content, end="", flush=True)
results = await llm.abatch(list_of_messages)  # batch, returns list
```

### Semaphore — rate limit control

```python
sem = asyncio.Semaphore(5)   # max 5 concurrent LLM calls

async def call_llm_safe(query):
    async with sem:           # waits if 5 are already running
        return await llm.ainvoke(query)
```

Without a semaphore: 100 parallel calls → burst rate limit error → all fail.  
With a semaphore: 100 calls flow through in controlled batches of 5.

### Streaming tokens over HTTP — why it matters

Without streaming: user waits 5 seconds staring at a loading spinner.  
With streaming: tokens appear in real time — perceived as ~10× faster.  
Pattern: `astream()` + FastAPI `StreamingResponse` + Server-Sent Events (SSE).

### 🧪 Try It Yourself — 12.1: Sequential vs async LLM calls — benchmark

1. Write `call_llm_sync(query)` using `llm.invoke()` — measures wall time
2. Write `call_llm_async(query)` using `await llm.ainvoke()` — measures wall time
3. Run 5 fixed queries **sequentially** — record total time
4. Run the same 5 queries with `asyncio.gather()` — record total time
5. Print the speedup ratio

Expected: async should be roughly N× faster where N is number of queries.

In [ ]:
# 🧪 YOUR CODE HERE — Day 12.1: Benchmark sequential vs async LLM calls
# from langchain_openai import ChatOpenAI
# llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

In [ ]:
# ✅ REFERENCE SOLUTION — Day 12.1: Sequential vs async benchmark
# ─────────────────────────────────────────────────────────────────────────────
# KEY INSIGHTS:
# 1. ainvoke() is the async equivalent of invoke() — same result, non-blocking.
# 2. gather() fans out all coroutines simultaneously — they all wait for I/O
#    concurrently. Total time ≈ slowest single call, not sum of all calls.
# 3. The speedup is proportional to N for I/O bound work. At 10 queries,
#    you typically see 5-8× improvement (real API variance adds noise).

import asyncio
import time
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

BENCHMARK_QUERIES = [
    "What is asyncio?",
    "What is the capital of Australia?",
    "Name 3 Python standard library modules.",
    "Explain Docker in one sentence.",
    "What does RAG stand for in AI?",
]


def call_llm_sync(query: str) -> str:
    response = llm.invoke([HumanMessage(content=query)])
    return response.content


async def call_llm_async(query: str) -> str:
    response = await llm.ainvoke([HumanMessage(content=query)])
    return response.content


# Sequential
print("=== Sequential (sync) ===")
t0 = time.perf_counter()
sync_results = [call_llm_sync(q) for q in BENCHMARK_QUERIES]
sync_time = time.perf_counter() - t0
print(f"Sequential time: {sync_time:.2f}s for {len(BENCHMARK_QUERIES)} queries")

# Async parallel
print("\n=== Async parallel (gather) ===")
t0 = time.perf_counter()
async_results = await asyncio.gather(*[call_llm_async(q) for q in BENCHMARK_QUERIES])
async_time = time.perf_counter() - t0
print(f"Async time: {async_time:.2f}s for {len(BENCHMARK_QUERIES)} queries")

speedup = sync_time / async_time
print(f"\n🚀 Speedup: {speedup:.1f}×  ({sync_time:.1f}s → {async_time:.1f}s)")

### 🧪 Try It Yourself — 12.2: Semaphore — rate limit control

1. Create `sem = asyncio.Semaphore(3)`
2. Write `call_llm_safe(query, sem)` that acquires the semaphore with `async with sem:` before calling `ainvoke`
3. Run 10 queries with this — observe max 3 run at once (add print timestamps to verify)
4. Compare time to uncapped `gather()` of 10

In [ ]:
# 🧪 YOUR CODE HERE — Day 12.2: Semaphore-controlled async LLM calls

In [ ]:
# ✅ REFERENCE SOLUTION — Day 12.2: Semaphore for concurrency control
# ─────────────────────────────────────────────────────────────────────────────
# KEY INSIGHTS:
# 1. Semaphore(N) = "at most N concurrent holders at any time".
#    When N slots are taken, new callers await until one is released.
# 2. This is the production pattern for LLM calls — protects against:
#    a) Rate limit errors (too many requests/minute)
#    b) Cost spikes (accidental fan-out of 1000 parallel calls)
# 3. Tune N based on your API tier's rate limit — start conservative, increase.

import asyncio
import time

MAX_CONCURRENT = 3
sem = asyncio.Semaphore(MAX_CONCURRENT)

active_count = 0   # for observation

async def call_llm_safe(query: str) -> str:
    """Call LLM with semaphore — max MAX_CONCURRENT calls at once."""
    global active_count
    async with sem:
        active_count += 1
        print(f"  [{time.perf_counter():.1f}s] Starting (active={active_count}): {query[:35]}...")
        result = await llm.ainvoke([HumanMessage(content=query)])
        active_count -= 1
        return result.content


TEN_QUERIES = BENCHMARK_QUERIES * 2  # 10 queries

print(f"=== 10 queries with Semaphore({MAX_CONCURRENT}) ===")
active_count = 0
t0 = time.perf_counter()
safe_results = await asyncio.gather(*[call_llm_safe(q) for q in TEN_QUERIES])
capped_time = time.perf_counter() - t0
print(f"\nTotal time with semaphore: {capped_time:.2f}s for 10 queries")
print(f"Max concurrent was capped at {MAX_CONCURRENT}")
print(f"All {len(safe_results)} results received.")

### 🧪 Try It Yourself — 12.3: Streaming LLM tokens with astream()

1. Use `llm.astream([HumanMessage(...)])` with `async for chunk in ...:`
2. Print each chunk's content as it arrives (no newline between — real streaming effect)
3. Measure: how long until the first token arrives vs total time?
4. Write an async generator `stream_response(query)` that yields chunks — this is the building block for SSE

In [ ]:
# 🧪 YOUR CODE HERE — Day 12.3: Streaming with astream()

In [ ]:
# ✅ REFERENCE SOLUTION — Day 12.3: Streaming tokens + async generator
# ─────────────────────────────────────────────────────────────────────────────
# KEY INSIGHTS:
# 1. astream() returns an async generator — you iterate with `async for`.
#    Each chunk is an AIMessageChunk with a `.content` string (partial token).
# 2. time_to_first_token is the latency users perceive. With streaming,
#    they see text immediately even though full response takes 3–5s.
# 3. An async generator that yields chunks is EXACTLY what FastAPI
#    StreamingResponse expects — this is the production SSE pattern.

import asyncio
import time
from typing import AsyncGenerator

print("=== Live token streaming ===")
t0 = time.perf_counter()
first_token_time = None

async for chunk in llm.astream([HumanMessage(content="Write a 3-sentence explanation of asyncio for a Python developer.")]):
    if first_token_time is None:
        first_token_time = time.perf_counter() - t0
    print(chunk.content, end="", flush=True)

total_time = time.perf_counter() - t0
print(f"\n\nTime to first token: {first_token_time:.2f}s")
print(f"Total streaming time: {total_time:.2f}s")


# Async generator — the building block for FastAPI SSE
async def stream_response(query: str) -> AsyncGenerator[str, None]:
    """Yield formatted SSE chunks. Used by FastAPI StreamingResponse."""
    async for chunk in llm.astream([HumanMessage(content=query)]):
        if chunk.content:
            yield f"data: {chunk.content}\n\n"   # SSE format


print("\n=== Async generator for SSE (first 5 chunks) ===")
count = 0
async for sse_chunk in stream_response("Name 3 Python built-in functions."):
    print(repr(sse_chunk[:50]))
    count += 1
    if count >= 5:
        print("  ... (truncated for demo)")
        break

print("\n→ FastAPI StreamingResponse wraps this generator:")
print("   from fastapi.responses import StreamingResponse")
print("   return StreamingResponse(stream_response(query), media_type='text/event-stream')")

### 🧪 Try It Yourself — 12.4: Retry with exponential backoff using tenacity

1. Install `tenacity` (async support)
2. Write a decorated `async def call_llm_with_retry(query)` using `@retry` from tenacity with:
   - `stop=stop_after_attempt(3)`
   - `wait=wait_exponential(multiplier=1, min=1, max=10)`
   - `retry=retry_if_exception_type(Exception)` but only retrying on rate limit errors
3. Simulate a rate limit error — verify the retry fires and backs off
4. Log each retry attempt with attempt number and wait time

In [ ]:
# 🧪 YOUR CODE HERE — Day 12.4: Async retry with tenacity

In [ ]:
# ✅ REFERENCE SOLUTION — Day 12.4: Async retry with exponential backoff
# ─────────────────────────────────────────────────────────────────────────────
# KEY INSIGHTS:
# 1. tenacity's @retry works with async functions — use it exactly like sync.
# 2. Exponential backoff: wait 1s → 2s → 4s → 8s... avoids thundering herd
#    when an API comes back up after a rate limit window.
# 3. `before_sleep=log_retry_attempt` is essential in prod — you need to know
#    when retries are happening (a spike means something is wrong upstream).
# 4. Only retry transient errors (rate limits, timeouts, 5xx). Never retry
#    auth errors (401) or bad input errors (400) — they won't recover.

from tenacity import (
    retry, stop_after_attempt, wait_exponential,
    retry_if_exception_type, before_sleep_log
)
import logging

logging.basicConfig(level=logging.WARNING)
logger = logging.getLogger("tenacity")

# Simulate a flaky API call
call_count = 0


async def flaky_llm_call(query: str) -> str:
    """Simulates an LLM API that fails the first 2 times with rate limit."""
    global call_count
    call_count += 1
    if call_count <= 2:
        print(f"  [attempt {call_count}] Simulated rate limit error")
        raise ConnectionError(f"429 Rate limit exceeded (attempt {call_count})")
    print(f"  [attempt {call_count}] Success!")
    return f"Response to: {query}"


@retry(
    stop=stop_after_attempt(4),
    wait=wait_exponential(multiplier=0.1, min=0.1, max=2),  # fast for demo; use min=1 in prod
    retry=retry_if_exception_type(ConnectionError),
    reraise=True,
)
async def call_llm_with_retry(query: str) -> str:
    """Call the LLM with automatic retry on connection/rate limit errors."""
    return await flaky_llm_call(query)


print("=== Retry with exponential backoff ===")
call_count = 0
result = await call_llm_with_retry("What is asyncio?")
print(f"\nFinal result: {result}")
print(f"Total attempts made: {call_count}")

### ✅ Day 12 Checkpoint

- [ ] You've benchmarked sequential vs async LLM calls — you have a real speedup number  
- [ ] You can explain why a semaphore is non-optional at scale  
- [ ] You have a working `stream_response()` async generator — you know how it plugs into FastAPI  
- [ ] You've implemented retry with exponential backoff  

**📝 Write LinkedIn Post #5 tonight** — open `02_linkedin_posts.md` → Post 5 (Days 12–13)

---

# 📅 Day 13 — Async Context Managers, Generators & Error Handling

**Time budget:** ~20 min concepts + 100 min hands-on

---

## 📖 Must-Read Docs

| Doc | What you'll learn | URL |
|---|---|---|
| httpx async | AsyncClient as context manager | https://www.python-httpx.org/async/ |
| Python asynccontextmanager | Write your own async context managers | https://docs.python.org/3/library/contextlib.html#contextlib.asynccontextmanager |
| asyncio.gather return_exceptions | Partial failure handling | https://docs.python.org/3/library/asyncio-task.html#asyncio.gather |

**Minimum read on a lazy day:** Read the httpx async page. Understand why `async with AsyncClient() as client:` is the correct pattern (connection pooling, clean teardown).

---

## 🧠 Core Concepts

### async with — async context managers

```python
# async context manager = resource with async setup + teardown
async with httpx.AsyncClient() as client:
    response = await client.get(url)   # client is set up (connection pool open)
    # ... use client ...
# client.__aexit__ called here — connections closed, resources freed
```

Use for: HTTP clients, DB connections, file handles, locks, rate limiters.

### async for — async generators

```python
async def stream_chunks(url):
    async with httpx.AsyncClient() as client:
        async with client.stream("GET", url) as response:
            async for chunk in response.aiter_bytes():
                yield chunk   # this makes it an async generator

async for chunk in stream_chunks(url):
    process(chunk)
```

### Partial failure handling with gather()

```python
# Default: gather() raises the FIRST exception, cancels remaining
results = await asyncio.gather(task1, task2, task3)  # if task2 fails → exception

# With return_exceptions=True: gather ALL results, failures are returned as exceptions
results = await asyncio.gather(task1, task2, task3, return_exceptions=True)
# results = ["ok", RuntimeError("task2 failed"), "ok"]
# You handle each — log failures, collect successes
```

### Silent failures — the async pitfall

```python
# ❌ BAD: creating a task and not awaiting it
asyncio.create_task(risky_fn())  # if it raises, the error is LOST silently

# ✅ GOOD: always store the task and handle errors
task = asyncio.create_task(risky_fn())
task.add_done_callback(lambda t: t.exception() and logger.error(t.exception()))
```

### 🧪 Try It Yourself — 13.1: Async HTTP client with httpx

1. Use `httpx.AsyncClient` as an async context manager
2. Fetch 3 public URLs concurrently using `gather()`
3. Print status codes and response times
4. Handle a connection error for one URL gracefully (don't crash the others)

In [ ]:
# 🧪 YOUR CODE HERE — Day 13.1: Async HTTP client with httpx
# import httpx
# async with httpx.AsyncClient() as client: ...

In [ ]:
# ✅ REFERENCE SOLUTION — Day 13.1: Async HTTP with httpx
# ─────────────────────────────────────────────────────────────────────────────
# KEY INSIGHTS:
# 1. AsyncClient() manages a connection pool — reuse it across requests.
#    Don't create a new client per request (expensive). Always use `async with`.
# 2. return_exceptions=True + isinstance check is the pattern for concurrent
#    requests where some may fail — you want successes even if some fail.
# 3. httpx is the async-native HTTP library for Python.
#    Don't use requests inside async code — it blocks the event loop.

import asyncio
import time
import httpx

URLS = [
    "https://httpbin.org/get",
    "https://httpbin.org/delay/1",
    "https://httpbin.org/status/200",
    "https://this-does-not-exist-xyz-12345.com",   # will fail
]


async def fetch_url(client: httpx.AsyncClient, url: str) -> dict:
    """Fetch a URL and return metadata. Raises on network error."""
    t0 = time.perf_counter()
    response = await client.get(url, timeout=5.0)
    elapsed = time.perf_counter() - t0
    return {"url": url, "status": response.status_code, "time": elapsed}


print("=== Concurrent HTTP requests with httpx ===")
t0 = time.perf_counter()

async with httpx.AsyncClient() as client:
    results = await asyncio.gather(
        *[fetch_url(client, url) for url in URLS],
        return_exceptions=True   # don't let one failure kill the others
    )

total = time.perf_counter() - t0

print(f"\nAll {len(URLS)} requests completed in {total:.2f}s:")
for r in results:
    if isinstance(r, Exception):
        print(f"  ❌ FAILED: {type(r).__name__}: {str(r)[:60]}")
    else:
        print(f"  ✅ {r['status']}  {r['time']:.2f}s  {r['url']}")

### 🧪 Try It Yourself — 13.2: Partial failure handling with gather(return_exceptions=True)

1. Write `async def score_query(query, idx)` — calls LLM but randomly fails for idx=2 and idx=5
2. Run 8 queries with `gather(return_exceptions=True)`
3. Split results: collect successes in a list, log failures to a list
4. Print summary: N succeeded, M failed, fail details
5. Demonstrate why `return_exceptions=False` is NOT what you want here

In [ ]:
# 🧪 YOUR CODE HERE — Day 13.2: Partial failure with return_exceptions=True

In [ ]:
# ✅ REFERENCE SOLUTION — Day 13.2: Partial failure handling
# ─────────────────────────────────────────────────────────────────────────────
# KEY INSIGHTS:
# 1. return_exceptions=True is essential in any production fan-out.
#    Without it: 1 failure cancels 7 successes. That's usually wrong.
# 2. isinstance(r, BaseException) is the check — not Exception, because
#    asyncio.CancelledError is a BaseException in Python 3.8+.
# 3. Logging failures with context (which query, which index) is critical —
#    otherwise you have 2 mysteriously missing results with no explanation.

import asyncio
import random

FAIL_AT_INDICES = {2, 5}

async def score_query(query: str, idx: int) -> dict:
    """Score a query. Simulates random failure at certain indices."""
    await asyncio.sleep(random.uniform(0.1, 0.3))  # simulate API latency
    if idx in FAIL_AT_INDICES:
        raise RuntimeError(f"Simulated API error for query [{idx}]: {query!r}")
    # Simulate an LLM score
    return {"idx": idx, "query": query, "score": round(random.uniform(2.5, 5.0), 2)}


TEST_QUERIES = [
    "What is asyncio?", "Explain LangGraph", "What is RAGAS?",
    "What is Docker?", "What is FastAPI?", "What is a semaphore?",
    "What is LangSmith?", "What is a guardrail?",
]

print("=== Partial failure handling with return_exceptions=True ===")
results = await asyncio.gather(
    *[score_query(q, i) for i, q in enumerate(TEST_QUERIES)],
    return_exceptions=True,
)

successes = [r for r in results if not isinstance(r, BaseException)]
failures  = [(i, r) for i, r in enumerate(results) if isinstance(r, BaseException)]

print(f"\n✅ Succeeded: {len(successes)}/{len(TEST_QUERIES)}")
for s in successes:
    print(f"   [{s['idx']}] {s['query'][:30]:<30}  score={s['score']}")

print(f"\n❌ Failed: {len(failures)}")
for idx, exc in failures:
    print(f"   [idx={idx}] {type(exc).__name__}: {exc}")

avg_score = sum(s['score'] for s in successes) / len(successes)
print(f"\nAverage score (successes only): {avg_score:.2f}")

### 🧪 Try It Yourself — 13.3: Write a reusable async_retry decorator

Write your own `async_retry` decorator (without tenacity) that:
1. Takes `max_attempts` and `backoff_factor` as parameters
2. Wraps an async function
3. On exception: waits `backoff_factor * (2 ** attempt)` seconds and retries
4. After `max_attempts`, re-raises the last exception
5. Logs each retry with attempt number

In [ ]:
# 🧪 YOUR CODE HERE — Day 13.3: Write async_retry decorator from scratch

In [ ]:
# ✅ REFERENCE SOLUTION — Day 13.3: async_retry decorator
# ─────────────────────────────────────────────────────────────────────────────
# KEY INSIGHTS:
# 1. A decorator returns a FUNCTION that wraps the original — same for async.
#    The wrapper must also be `async def` to properly preserve the awaitable interface.
# 2. functools.wraps(fn) preserves __name__, __doc__ — important for debugging
#    and tracing (makes the wrapper transparent in stack traces).
# 3. This is useful when you can't add tenacity as a dependency, or when
#    you need behaviour tenacity doesn't support out of the box.

import asyncio
import functools
import logging
from typing import Callable, Any

logger = logging.getLogger(__name__)


def async_retry(
    max_attempts: int = 3,
    backoff_factor: float = 0.5,
    exceptions: tuple = (Exception,),
):
    """Decorator: retry an async function with exponential backoff."""
    def decorator(fn: Callable) -> Callable:
        @functools.wraps(fn)
        async def wrapper(*args: Any, **kwargs: Any) -> Any:
            last_exc = None
            for attempt in range(1, max_attempts + 1):
                try:
                    return await fn(*args, **kwargs)
                except exceptions as exc:
                    last_exc = exc
                    if attempt == max_attempts:
                        break
                    wait = backoff_factor * (2 ** (attempt - 1))
                    print(f"  [{fn.__name__}] attempt {attempt} failed: {exc}. Retrying in {wait:.1f}s...")
                    await asyncio.sleep(wait)
            raise last_exc  # re-raise after all attempts exhausted
        return wrapper
    return decorator


# Test it
attempt_counter = 0

@async_retry(max_attempts=3, backoff_factor=0.2)
async def unreliable_api_call(value: str) -> str:
    global attempt_counter
    attempt_counter += 1
    if attempt_counter < 3:
        raise ConnectionError(f"Transient error on attempt {attempt_counter}")
    return f"Success on attempt {attempt_counter}: {value}"


attempt_counter = 0
print("=== async_retry decorator test ===")
result = await unreliable_api_call("test_payload")
print(f"Final result: {result}")

# Test exhaustion
attempt_counter = 0

@async_retry(max_attempts=2, backoff_factor=0.1)
async def always_fails() -> str:
    raise RuntimeError("This always fails")

print("\n=== Testing exhaustion ===")
try:
    await always_fails()
except RuntimeError as e:
    print(f"Expected error after all retries: {e}")

### ✅ Day 13 Checkpoint

- [ ] You can explain why `async with AsyncClient()` beats creating a new client per request  
- [ ] You know the difference between `return_exceptions=True` and `False` in gather — and why it matters  
- [ ] You have a working `async_retry` decorator you could drop into any codebase  
- [ ] You know how to detect and log silent failures from fire-and-forget tasks  

---

# 📅 Day 14 — Apply Everything: Async LangGraph Agent

**Time budget:** ~10 min concepts + 110 min hands-on  
**📝 Write LinkedIn Post #6 tonight** — see `02_linkedin_posts.md` (Sprint wrap-up post)

---

## 📖 Must-Read Docs

| Doc | What you'll learn | URL |
|---|---|---|
| LangGraph async support | ainvoke, astream_events | https://langchain-ai.github.io/langgraph/how-tos/streaming/ |
| astream_events | Event-based streaming format | https://langchain-ai.github.io/langgraph/how-tos/streaming-events-from-within-tools/ |
| pytest-asyncio | Testing async code | https://pytest-asyncio.readthedocs.io/en/latest/ |

**Minimum read on a lazy day:** LangGraph streaming how-to. Understand `astream_events` and what event types it emits (`on_chat_model_stream`, `on_chain_start`, etc.).

---

## 🧠 Core Concepts

### LangGraph async support

```python
# All LangGraph invocation methods have async equivalents
result = await graph.ainvoke(state)              # async invoke
async for event in graph.astream_events(state, version="v2"):  # event streaming
    if event["event"] == "on_chat_model_stream":
        print(event["data"]["chunk"].content, end="")
```

### Async nodes

```python
# Sync node — fine for most use cases
def my_node(state: MyState) -> dict:
    return {"result": llm.invoke(...)}

# Async node — needed when using ainvoke or parallel tool calls inside a node
async def my_async_node(state: MyState) -> dict:
    result = await llm.ainvoke(...)   # non-blocking
    return {"result": result}
```

### FastAPI + async LangGraph — the full stack

```python
@app.get("/stream")
async def stream_endpoint(query: str):
    async def event_generator():
        async for event in graph.astream_events({"query": query}, version="v2"):
            if event["event"] == "on_chat_model_stream":
                token = event["data"]["chunk"].content
                if token:
                    yield f"data: {token}\n\n"
    return StreamingResponse(event_generator(), media_type="text/event-stream")
```

### Load testing with httpx async client

Simulate concurrent users by running N coroutines in parallel — each makes a request to your FastAPI server. Measure throughput (requests/second) and p95 latency.

```python
async def load_test(url, n_requests, max_concurrent):
    sem = asyncio.Semaphore(max_concurrent)
    async with httpx.AsyncClient() as client:
        tasks = [make_request(client, url, sem) for _ in range(n_requests)]
        results = await asyncio.gather(*tasks, return_exceptions=True)
    # analyse results
```

### 🧪 Try It Yourself — 14.1: Refactor Day 3 supervisor graph to fully async

Take the supervisor multi-agent pattern from Day 3 and:
1. Change all node functions to `async def`
2. Replace `llm.invoke()` calls with `await llm.ainvoke()`
3. In the tools/research node: if there are 2 independent tool calls, parallelise them with `asyncio.gather()`
4. Run with `await graph.ainvoke(state)` — verify same output
5. Print timing comparison: sync version vs async version

In [ ]:
# 🧪 YOUR CODE HERE — Day 14.1: Async supervisor multi-agent graph
# Rebuild the Day 3 supervisor with async nodes
# Use await graph.ainvoke() at the end

In [ ]:
# ✅ REFERENCE SOLUTION — Day 14.1: Async multi-agent graph
# ─────────────────────────────────────────────────────────────────────────────
# KEY INSIGHTS:
# 1. Converting sync nodes to async is mechanical — just add `async def` and `await`.
#    LangGraph handles both sync and async nodes transparently.
# 2. The real win is inside nodes with PARALLEL tool calls:
#    `await asyncio.gather(call_a(), call_b())` instead of sequential.
# 3. `await graph.ainvoke()` is needed when the graph contains async nodes.
#    Using sync `graph.invoke()` on a graph with async nodes will fail.

import asyncio
from typing import TypedDict, Annotated, Literal, Optional
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage, SystemMessage, AIMessage

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

class AsyncSupervisorState(TypedDict):
    messages: Annotated[list, add_messages]
    next_call: Optional[Literal["research", "writer", "FINISH"]]


SUPERVISOR_SYSTEM = (
    "You are a supervisor. Review the conversation and decide who to call next: "
    "'research' (to gather information), 'writer' (to draft content), "
    "or 'FINISH' (when the task is complete). Respond with ONE word only."
)
RESEARCH_SYSTEM = "You are a research assistant. Provide key factual bullet points on the topic from the last message."
WRITER_SYSTEM   = "You are a writer. Use the research points in the conversation to write a clear, concise paragraph."


async def supervisor_node(state: AsyncSupervisorState) -> dict:
    msgs = [SystemMessage(content=SUPERVISOR_SYSTEM)] + state["messages"][-3:]
    result = await llm.ainvoke(msgs)
    decision = result.content.strip().lower()
    if "research" in decision:
        return {"next_call": "research"}
    elif "writer" in decision or "write" in decision:
        return {"next_call": "writer"}
    return {"next_call": "FINISH"}


async def research_node(state: AsyncSupervisorState) -> dict:
    msgs = [SystemMessage(content=RESEARCH_SYSTEM)] + [state["messages"][0]]
    result = await llm.ainvoke(msgs)
    return {"messages": [result]}


async def writer_node(state: AsyncSupervisorState) -> dict:
    msgs = [SystemMessage(content=WRITER_SYSTEM)] + state["messages"][-2:]
    result = await llm.ainvoke(msgs)
    return {"messages": [result]}


def router(state: AsyncSupervisorState) -> str:
    return state["next_call"] if state["next_call"] != "FINISH" else END


builder = StateGraph(AsyncSupervisorState)
builder.add_node("supervisor", supervisor_node)
builder.add_node("research",   research_node)
builder.add_node("writer",     writer_node)
builder.set_entry_point("supervisor")
builder.add_conditional_edges("supervisor", router)
builder.add_edge("research", "supervisor")
builder.add_edge("writer",   "supervisor")
async_graph = builder.compile()


import time
print("=== Async supervisor graph ===")
t0 = time.perf_counter()
result = await async_graph.ainvoke({
    "messages": [HumanMessage(content="Write a short paragraph about quantum computing.")],
    "next_call": None,
})
elapsed = time.perf_counter() - t0
print(f"\nFinal output ({elapsed:.1f}s):")
print(result["messages"][-1].content)

### 🧪 Try It Yourself — 14.2: Stream LangGraph events with astream_events()

1. Call `graph.astream_events(state, version="v2")` on your async graph
2. Print a formatted log of events as they arrive:
   - `on_chain_start` / `on_chain_end` → `[node_name] started/finished`
   - `on_chat_model_stream` → print the token inline
3. Measure time from start to first LLM token (time-to-first-token for the whole graph)

In [ ]:
# 🧪 YOUR CODE HERE — Day 14.2: astream_events() with event inspection

In [ ]:
# ✅ REFERENCE SOLUTION — Day 14.2: astream_events() event inspection
# ─────────────────────────────────────────────────────────────────────────────
# KEY INSIGHTS:
# 1. astream_events version="v2" is the current standard — always use v2.
# 2. Events have an 'event' type and a 'name' field:
#    event='on_chain_start', name='supervisor' → supervisor node started
#    event='on_chat_model_stream' → a token arrived from the LLM
# 3. Filtering on event type lets you build rich streaming UIs:
#    - Show "Thinking..." on on_chain_start for known nodes
#    - Stream tokens on on_chat_model_stream
#    - Show "Done" on on_chain_end

import asyncio
import time

print("=== astream_events() — watching the graph think ===\n")
t0 = time.perf_counter()
first_token_time = None
current_tokens = []

KNOWN_NODES = {"supervisor", "research", "writer"}

async for event in async_graph.astream_events(
    {"messages": [HumanMessage(content="Explain what a transformer model is.")], "next_call": None},
    version="v2"
):
    etype = event["event"]
    ename = event.get("name", "")
    elapsed = time.perf_counter() - t0

    if etype == "on_chain_start" and ename in KNOWN_NODES:
        if current_tokens:
            print()  # newline after token stream
            current_tokens.clear()
        print(f"[{elapsed:.1f}s] ▶ Node '{ename}' started")

    elif etype == "on_chain_end" and ename in KNOWN_NODES:
        if current_tokens:
            print()
            current_tokens.clear()
        print(f"[{elapsed:.1f}s] ■ Node '{ename}' finished")

    elif etype == "on_chat_model_stream":
        token = event["data"]["chunk"].content
        if token:
            if first_token_time is None:
                first_token_time = elapsed
            print(token, end="", flush=True)
            current_tokens.append(token)

total = time.perf_counter() - t0
print(f"\n\n[Summary]")
print(f"  Time to first token: {first_token_time:.2f}s")
print(f"  Total run time     : {total:.2f}s")

### 🧪 Try It Yourself — 14.3: Write pytest-asyncio tests for 2 nodes

1. Install `pytest-asyncio`
2. Write a `test_nodes.py` file (created inline below) with tests for:
   - `test_research_node_returns_messages`: verify research_node adds a message to state
   - `test_supervisor_routes_to_research`: give the supervisor a fresh question, verify it routes to research (not FINISH)
3. Use `@pytest.mark.asyncio` decorator
4. Run with the `!pytest` command and verify both tests pass

In [ ]:
# 🧪 YOUR CODE HERE — Day 14.3: Write pytest-asyncio tests
# Create test_nodes.py with 2 async test functions
# Then run: !pytest test_nodes.py -v

In [ ]:
# ✅ REFERENCE SOLUTION — Day 14.3: pytest-asyncio test file
# ─────────────────────────────────────────────────────────────────────────────
# KEY INSIGHTS:
# 1. @pytest.mark.asyncio lets pytest run coroutines as test functions.
#    Mark tests with it OR set asyncio_mode="auto" in pytest.ini.
# 2. Testing nodes in isolation (not the whole graph) is fast and reliable —
#    you control the input state exactly, no graph routing uncertainty.
# 3. Mock the LLM in unit tests — you're testing the NODE LOGIC, not OpenAI.
#    Use actual LLM calls only in integration tests.

from pathlib import Path

TEST_FILE = '''
"""Async tests for the Day 14 multi-agent graph nodes."""
import pytest
from unittest.mock import AsyncMock, patch, MagicMock
from langchain_core.messages import AIMessage, HumanMessage

# We import from the notebook context — in a real project, import from your module
# from src.agents import research_node, supervisor_node, AsyncSupervisorState

# ── Minimal inline implementations for self-contained test file ───────────────
from typing import TypedDict, Annotated, Optional, Literal
from langgraph.graph.message import add_messages
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

class AsyncSupervisorState(TypedDict):
    messages: Annotated[list, add_messages]
    next_call: Optional[Literal["research", "writer", "FINISH"]]

RESEARCH_SYSTEM = "You are a research assistant. Provide key factual bullet points."

async def research_node(state: AsyncSupervisorState) -> dict:
    from langchain_core.messages import SystemMessage
    msgs = [SystemMessage(content=RESEARCH_SYSTEM)] + [state["messages"][0]]
    result = await llm.ainvoke(msgs)
    return {"messages": [result]}

async def supervisor_node(state: AsyncSupervisorState) -> dict:
    from langchain_core.messages import SystemMessage
    SUPERVISOR_SYSTEM = (
        "Decide: research, writer, or FINISH. Respond with one word."
    )
    msgs = [SystemMessage(content=SUPERVISOR_SYSTEM)] + state["messages"][-3:]
    result = await llm.ainvoke(msgs)
    decision = result.content.strip().lower()
    if "research" in decision:
        return {"next_call": "research"}
    elif "writer" in decision or "write" in decision:
        return {"next_call": "writer"}
    return {"next_call": "FINISH"}


# ── Tests ─────────────────────────────────────────────────────────────────────

@pytest.mark.asyncio
async def test_research_node_returns_messages():
    """research_node should add exactly one AIMessage to state."""
    mock_response = AIMessage(content="• Bullet point 1\n• Bullet point 2")

    with patch.object(llm, "ainvoke", new_callable=AsyncMock, return_value=mock_response):
        state: AsyncSupervisorState = {
            "messages": [HumanMessage(content="Explain asyncio")],
            "next_call": None,
        }
        result = await research_node(state)

    assert "messages" in result
    assert len(result["messages"]) == 1
    assert isinstance(result["messages"][0], AIMessage)
    assert "Bullet" in result["messages"][0].content


@pytest.mark.asyncio
async def test_supervisor_routes_to_research_for_fresh_question():
    """Supervisor should route to research when no research has been done yet."""
    mock_response = AIMessage(content="research")  # simulate LLM saying research

    with patch.object(llm, "ainvoke", new_callable=AsyncMock, return_value=mock_response):
        state: AsyncSupervisorState = {
            "messages": [HumanMessage(content="Tell me about quantum computing")],
            "next_call": None,
        }
        result = await supervisor_node(state)

    assert result["next_call"] == "research"


@pytest.mark.asyncio
async def test_supervisor_routes_to_finish_when_done():
    """Supervisor should route to FINISH when it decides the task is complete."""
    mock_response = AIMessage(content="FINISH")

    with patch.object(llm, "ainvoke", new_callable=AsyncMock, return_value=mock_response):
        state: AsyncSupervisorState = {
            "messages": [
                HumanMessage(content="Write about AI"),
                AIMessage(content="Some research notes..."),
                AIMessage(content="Here is the final paragraph..."),
            ],
            "next_call": None,
        }
        result = await supervisor_node(state)

    assert result["next_call"] == "FINISH"
'''

test_path = Path("test_nodes.py")
test_path.write_text(TEST_FILE.strip())
print(f"✅ test_nodes.py written to {test_path.resolve()}")
print("\nRun tests:")
print("  pip install pytest pytest-asyncio")
print("  pytest test_nodes.py -v")

In [ ]:
# Run the tests — expects pytest and pytest-asyncio installed
# !pip install -q pytest pytest-asyncio
# !pytest test_nodes.py -v --asyncio-mode=auto

### 📋 Day 14.4: FastAPI streaming endpoint (run in terminal)

The FastAPI server needs to run as a separate process — paste this into `async_agent_server.py` and run it from a terminal.

```python
# async_agent_server.py
import asyncio
import os
from dotenv import load_dotenv
load_dotenv()

from fastapi import FastAPI
from fastapi.responses import StreamingResponse
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage
from langgraph.graph import StateGraph, END
from typing import TypedDict

app = FastAPI(title="Async LangGraph Agent")
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

class SimpleState(TypedDict):
    query: str
    response: str

async def llm_node(state: SimpleState) -> dict:
    result = await llm.ainvoke([HumanMessage(content=state["query"])])
    return {"response": result.content}

builder = StateGraph(SimpleState)
builder.add_node("llm", llm_node)
builder.set_entry_point("llm")
builder.add_edge("llm", END)
graph = builder.compile()

@app.post("/invoke")
async def invoke(query: str):
    result = await graph.ainvoke({"query": query, "response": ""})
    return {"response": result["response"]}

@app.get("/stream")
async def stream(query: str):
    async def event_generator():
        async for event in graph.astream_events(
            {"query": query, "response": ""}, version="v2"
        ):
            if event["event"] == "on_chat_model_stream":
                token = event["data"]["chunk"].content
                if token:
                    yield f"data: {token}\n\n"
    return StreamingResponse(event_generator(), media_type="text/event-stream")

# Run: uvicorn async_agent_server:app --host 0.0.0.0 --port 8000
# Test: curl -N "http://localhost:8000/stream?query=What+is+asyncio"
```

**Terminal commands:**
```bash
# Start server
uvicorn async_agent_server:app --host 0.0.0.0 --port 8000

# Test invoke endpoint
curl -X POST "http://localhost:8000/invoke?query=What+is+asyncio"

# Test streaming endpoint (watch tokens arrive in real time)
curl -N "http://localhost:8000/stream?query=Explain+LangGraph+concisely"
```

### ✅ Day 14 Checkpoint — Sprint Complete

- [ ] All nodes are `async def` — you understand why and when it matters  
- [ ] `astream_events()` is working — you can see token streaming in the terminal  
- [ ] FastAPI `/invoke` and `/stream` endpoints work — you tested both with curl  
- [ ] 3 pytest tests pass with mocked LLM calls  
- [ ] You can explain what `return_exceptions=True` does and why it matters  

**📝 Write LinkedIn Post #6 tonight** — open `02_linkedin_posts.md` → Post 6 (Sprint wrap-up)

---

## 🏁 Days 11–14 Sprint Complete — Full Sprint Done

**What you've built across all 14 days:**

| Deliverable | Days | Skills |
|---|---|---|
| Multi-agent LangGraph system — deployed on AWS | 1–5 | LangGraph, multi-agent design, AWS |
| RAGAS + custom eval + CI/CD gate | 6–10 | LLM eval, LangSmith, GitHub Actions |
| 5 async patterns: gather, tasks, queue, timeout, executor | 11 | asyncio fundamentals |
| Async LLM benchmark — 5× speedup measured | 12 | async LLM, semaphore, streaming, tenacity |
| Reusable async utilities: retry, partial failure, httpx pool | 13 | async context managers, generators |
| Async LangGraph agent — streamed, tested, FastAPI-served | 14 | Full async stack |
| 6 LinkedIn posts | All | Personal brand, thought leadership |

**You went from "I've used LangChain" to building production-grade agentic AI systems with evaluation discipline, observability, and async engineering. That's the job.**